# CIM SoC 板上测试 — 通用配置版

支持 MLP / LeNet-5，可选手动选择 bitstream、data_dir、DMA、batch 推理。

包含：AXI连通性 → 单图验证+Profiler → 批量正确率+耗时统计

---

In [ ]:
# ====================================================================
# 0. 配置参数（改这里切换测试）
# ====================================================================
BITSTREAM = "arm/cim_soc.xsa"   # 或 "cim_soc.bit" (PYNQ v2.x)
MODEL     = "lenet5"            # "mlp" 或 "lenet5"
DATA_DIR  = None                # None=自动选择 (mlp → mnist_real_data, lenet5 → lenet5_data)
USE_DMA   = True                # DMA stream 数据通路
BATCH     = True                # predict_batch 层间批处理
NO_FUSION = False               # True 禁用 Phase C Layer Fusion (A/B 对比)
N_IMAGES  = 200                 # 批量测试图片数（上限取决于 test_images/ 文件数）

if DATA_DIR is None:
    DATA_DIR = "mnist_real_data" if MODEL == "mlp" else "lenet5_data"

print(f"Config: MODEL={MODEL}, BITSTREAM={BITSTREAM}, DATA_DIR={DATA_DIR}")
print(f"         USE_DMA={USE_DMA}, BATCH={BATCH}, NO_FUSION={NO_FUSION}, N_IMAGES={N_IMAGES}")

In [ ]:
# ====================================================================
# 1. 导入 + 加载 bitstream
# ====================================================================
import numpy as np
import glob, os, time, sys
sys.path.insert(0, os.path.dirname(os.path.abspath("")) or ".")

from cim_driver import (
    CIMDriver, CIMModel, _make_run_id,
    weight_to_chunks, bias_to_u32,
    _BASE, _MMIO_SIZE,
)
import golden_model as gm

drv = CIMDriver(BITSTREAM, use_dma=USE_DMA)
print(f"Driver ready: use_dma={drv.use_dma}")

In [ ]:
# ====================================================================
# 2. AXI 连通性测试
# ====================================================================
from pynq import MMIO

mmio = MMIO(_BASE, _MMIO_SIZE)
print("=== AXI Connectivity Test ===")
status = mmio.read(0x004)
print(f"  STATUS = 0x{status:08x}")
mmio.write(0x010, 784)
rb = mmio.read(0x010)
print(f"  IN_DIM write=784, readback={rb}  {'PASS' if rb == 784 else 'FAIL'}")
assert rb == 784, "AXI read/write failed!"
print("AXI OK.")

In [ ]:
# ====================================================================
# 3. 加载模型
# ====================================================================
# --- helpers (from benchmark_e2e.py) ---
def read_hex_u32(path):
    with open(path) as f:
        return [int(l.strip(), 16) for l in f if l.strip()]

def read_hex_u8(path):
    with open(path) as f:
        return [int(l.strip(), 16) & 0xFF for l in f if l.strip()]


model = CIMModel(drv)

if MODEL == "lenet5":
    # LeNet-5: load from npz
    d = np.load(f"{DATA_DIR}/lenet5_qparams.npz")

    model.add_conv(d["conv1_weight"], d["conv1_bias"],
                   zp=int(d["conv1_zp"]), mult=int(d["conv1_mult"]),
                   shift=int(d["conv1_shift"]), stride=1, padding=0, relu=True)
    model.add_pool(2, 2)

    model.add_conv(d["conv2_weight"], d["conv2_bias"],
                   zp=int(d["conv2_zp"]), mult=int(d["conv2_mult"]),
                   shift=int(d["conv2_shift"]), stride=1, padding=0, relu=True)
    model.add_pool(2, 2)

    model.add_fc(256, 120, weight_to_chunks(d["fc3_weight"]), bias_to_u32(d["fc3_bias"]),
                 zp=int(d["fc3_zp"]), mult=int(d["fc3_mult"]), shift=int(d["fc3_shift"]),
                 relu=True, weight_int8=d["fc3_weight"], bias_int32=d["fc3_bias"])
    model.add_fc(120, 84, weight_to_chunks(d["fc4_weight"]), bias_to_u32(d["fc4_bias"]),
                 zp=int(d["fc4_zp"]), mult=int(d["fc4_mult"]), shift=int(d["fc4_shift"]),
                 relu=True, weight_int8=d["fc4_weight"], bias_int32=d["fc4_bias"])
    model.add_fc(84, 10, weight_to_chunks(d["fc5_weight"]), bias_to_u32(d["fc5_bias"]),
                 zp=int(d["fc5_zp"]), mult=int(d["fc5_mult"]), shift=int(d["fc5_shift"]),
                 relu=False, weight_int8=d["fc5_weight"], bias_int32=d["fc5_bias"])
    input_shape = (1, 28, 28)

else:
    # MLP: load from hex files
    qp_path = f"{DATA_DIR}/quant_params.hex"
    with open(qp_path) as f:
        vals = [int(l.strip(), 16) for l in f if l.strip()]
    fc1_mult, fc1_shift, fc2_mult, fc2_shift = vals[:4]

    zp_path = f"{DATA_DIR}/zero_points.hex"
    if os.path.exists(zp_path):
        with open(zp_path) as f:
            zp_vals = [int(l.strip(), 16) for l in f if l.strip()]
        fc1_zp = zp_vals[0] if zp_vals else -128
        fc2_zp = zp_vals[1] if len(zp_vals) > 1 else -128
    else:
        fc1_zp = fc2_zp = -128

    fc1_chunks = read_hex_u32(f"{DATA_DIR}/fc1_weight_tiles.hex")
    fc2_chunks = read_hex_u32(f"{DATA_DIR}/fc2_weight_tiles.hex")
    fc1_bias   = read_hex_u32(f"{DATA_DIR}/fc1_bias.hex")
    fc2_bias   = read_hex_u32(f"{DATA_DIR}/fc2_bias.hex")

    model.add_fc(784, 128, fc1_chunks, fc1_bias,
                 zp=fc1_zp, mult=fc1_mult, shift=fc1_shift, relu=True)
    model.add_fc(128, 10, fc2_chunks, fc2_bias,
                 zp=fc2_zp, mult=fc2_mult, shift=fc2_shift, relu=False)
    input_shape = (784,)

if NO_FUSION:
    model.use_fusion = False

# Show packing info
for i, layer in enumerate(model.layers):
    if layer.get("type") == "conv":
        p = layer.get("_packed")
        k = p["k_pack"] if p else 1
        print(f"  Layer {i} ({layer['C_in']}ch {layer['K_h']}x{layer['K_w']}->{layer['C_out']}ch): "
              f"col_len={layer['col_len']}, k_pack={k}")

print(f"\nModel: {len(model.layers)} layers loaded. use_fusion={model.use_fusion}")

In [ ]:
# ====================================================================
# 4. 单图推理 + bit-exact 验证 + Profiler
# ====================================================================
img_dir = f"{DATA_DIR}/test_images"
img_path = sorted(glob.glob(f"{img_dir}/img_????.hex"))[0]
img_u8 = np.array(read_hex_u8(img_path), dtype=np.uint8)
label = int(open(img_path.replace(".hex", "_label.txt")).read().strip())
print(f"Test image: {os.path.basename(img_path)}, label={label}\n")

print(f"=== Single Image {MODEL.upper()} (verify=True, profile=True) ===")
pred, logits, prof = model.predict(
    img_u8.reshape(input_shape),
    verbose=True, verify=True, profile=True,
)

print(f"\nHW pred={pred}, label={label}, {'CORRECT' if pred == label else 'WRONG'}")

# Profiler table
print(f"\n--- Profiler (per-image) ---")
header = f"{'Layer':<32s} {'n_mvm':>6s} {'k':>4s} {'setup':>8s} {'load_x':>8s} {'compute':>8s} {'read':>8s} {'total':>8s}"
print(header)
print("-" * len(header))
for lp in prof["layers"]:
    if lp["type"] == "pool":
        print(f"{lp['name']:<32s} {'--':>6s} {'--':>4s} {'--':>8s} {'--':>8s} {'--':>8s} {'--':>8s} {lp['total_ms']:>7.1f}ms")
    else:
        print(f"{lp['name']:<32s} {lp.get('n_mvm',0):>6d} {lp.get('k_pack',1):>4d} "
              f"{lp.get('setup_ms',0):>7.1f}ms {lp.get('load_x_ms',0):>7.1f}ms "
              f"{lp.get('compute_ms',0):>7.1f}ms {lp.get('read_out_ms',0):>7.1f}ms "
              f"{lp['total_ms']:>7.1f}ms")
print(f"{'TOTAL':<32s} {'':>6s} {'':>4s} {'':>8s} {'':>8s} {'':>8s} {'':>8s} {prof['total_ms']:>7.1f}ms")

In [ ]:
# ====================================================================
# 5. 批量推理：N 张图，统计正确率 + 平均耗时
# ====================================================================
import csv
from datetime import datetime

img_files = sorted(glob.glob(f"{img_dir}/img_????.hex"))
n = min(N_IMAGES, len(img_files))
img_files = img_files[:n]
print(f"=== Batch Test: {n} images ===\n")

# Load all images
all_images = []
all_labels = []
all_names = []
for img_path in img_files:
    name = os.path.basename(img_path).replace(".hex", "")
    img_u8 = np.array(read_hex_u8(img_path), dtype=np.uint8)
    label = int(open(f"{img_dir}/{name}_label.txt").read().strip())
    all_images.append(img_u8.reshape(input_shape))
    all_labels.append(label)
    all_names.append(name)

correct = 0
wrong_list = []
t_start = time.time()

if BATCH and USE_DMA:
    # Layer-wise batching with DMA ping-pong + fusion
    results, batch_prof = model.predict_batch(all_images, profile=True)
    for (pred, _), label, name in zip(results, all_labels, all_names):
        if pred == label:
            correct += 1
        else:
            wrong_list.append((name, pred, label))

    # Batch profiler
    total_ms = batch_prof.get("total_ms", 0)
    n_img = batch_prof.get("n_images", 1)
    per_img = total_ms / n_img
    print(f"\n--- Batch Profiler (per-image avg) ---")
    agg = {}
    for l in batch_prof.get("layers", []):
        for k in ("im2col_ms", "pack_ms", "setup_ms", "load_x_ms", "compute_ms", "read_out_ms", "pool_ms"):
            agg[k] = agg.get(k, 0) + l.get(k, 0)
    for k, v in agg.items():
        if v > 0.005:
            print(f"  {k:<30s} {v:8.2f}ms  {v/per_img*100:5.1f}%")
    print(f"  {'TOTAL per image':<30s} {per_img:8.2f}ms")
else:
    # Sequential per-image
    for name, img, label in zip(all_names, all_images, all_labels):
        pred, _ = model.predict(img)
        if pred == label:
            correct += 1
        else:
            wrong_list.append((name, pred, label))

elapsed = time.time() - t_start

# --- Results ---
total_s    = elapsed
ms_per_img = total_s / n * 1000
fps        = n / total_s
accuracy   = correct / n * 100

print(f"\n{'='*60}")
print(f"  Model={MODEL}  n={n}  total={total_s:.2f}s  ms/img={ms_per_img:.1f}  fps={fps:.2f}  acc={correct}/{n} ({accuracy:.1f}%)")
print(f"{'='*60}")

if wrong_list:
    print(f"\nWrong ({len(wrong_list)}):")
    for name, pred, label in wrong_list:
        print(f"  {name}: pred={pred} label={label}")

# --- CSV export ---
os.makedirs("results", exist_ok=True)
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_path = f"results/benchmark_{MODEL}_{ts}.csv"
with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["model", "n_img", "total_s", "ms_per_img", "fps", "correct", "accuracy_pct"])
    writer.writerow([MODEL, n, f"{total_s:.3f}", f"{ms_per_img:.2f}", f"{fps:.3f}", correct, f"{accuracy:.2f}"])
print(f"\nCSV saved: {csv_path}")